# Компьютерное зрение для поиска линейных кандидатов на разрывные нарушения

Ноутбук построен как воспроизводимый baseline без потерянных JPEG-файлов и без искусственно восстановленной разметки. Обработка идёт непосредственно из SEG-Y: нормализация → подавление шума → Canny → вероятностное преобразование Хафа → фильтрация линий по углу.

## Ограничение исходного ML-подхода

Старые JPEG-фрагменты и их разметка утеряны, поэтому корректно воспроизвести обучение CNN и оценить качество
модели невозможно. Обучать сеть на заново придуманных метках было бы методически хуже, чем явно признать
отсутствие ground truth. Поэтому основная версия ниже использует классическое компьютерное зрение и выдаёт
**кандидатов**, а не готовую геологическую интерпретацию.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

!pip -q install segyio opencv-python-headless

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import segyio

SEGY_PATH = Path('/content/drive/MyDrive/cw_data/TrainingData_Image.segy')
N_TRACES = 700

# Параметры компьютерного зрения.
GAUSSIAN_KERNEL = (5, 5)
CANNY_LOW = 40
CANNY_HIGH = 120
HOUGH_THRESHOLD = 35
MIN_LINE_LENGTH = 35
MAX_LINE_GAP = 12
MIN_ABS_ANGLE_DEG = 20.0
MAX_ABS_ANGLE_DEG = 80.0

if not SEGY_PATH.exists():
    raise FileNotFoundError(f'SEG-Y не найден: {SEGY_PATH}')

## 1. Загрузка двух контрольных фрагментов

In [ ]:
def load_trace_range(path: Path, start: int, stop: int) -> np.ndarray:
    """Читает диапазон и возвращает (sample, trace)."""
    with segyio.open(str(path), 'r', ignore_geometry=True) as segy:
        total = len(segy.trace)
        start = max(0, start)
        stop = min(stop, total)
        traces = np.stack([
            np.asarray(segy.trace[i], dtype=np.float32)
            for i in range(start, stop)
        ])
    return traces.T


def load_first_last(path: Path, n_traces: int) -> dict[str, np.ndarray]:
    with segyio.open(str(path), 'r', ignore_geometry=True) as segy:
        total = len(segy.trace)
    count = min(n_traces, total)
    return {
        'Первые трассы': load_trace_range(path, 0, count),
        'Последние трассы': load_trace_range(path, total - count, total),
    }


sections = load_first_last(SEGY_PATH, N_TRACES)
for name, section in sections.items():
    print(name, section.shape)

## 2. Перевод сейсмического разреза в изображение

In [ ]:
def seismic_to_uint8(section: np.ndarray, clip_percentile: float = 99.0) -> np.ndarray:
    """Устойчиво переводит амплитуды в 8-bit grayscale для OpenCV."""
    limit = np.percentile(np.abs(section), clip_percentile)
    if limit <= np.finfo(np.float32).eps:
        return np.zeros(section.shape, dtype=np.uint8)

    clipped = np.clip(section, -limit, limit)
    normalized = (clipped + limit) / (2.0 * limit)
    return np.round(normalized * 255).astype(np.uint8)


def edge_map(image: np.ndarray) -> np.ndarray:
    blurred = cv2.GaussianBlur(image, GAUSSIAN_KERNEL, sigmaX=0)
    return cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH, L2gradient=True)

## 3. Линии-кандидаты по преобразованию Хафа

In [ ]:
def line_angle_deg(x1: int, y1: int, x2: int, y2: int) -> float:
    """Угол линии относительно горизонтальной оси изображения."""
    return float(np.degrees(np.arctan2(y2 - y1, x2 - x1)))


def detect_candidate_lines(edges: np.ndarray) -> list[tuple[int, int, int, int, float]]:
    raw_lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=HOUGH_THRESHOLD,
        minLineLength=MIN_LINE_LENGTH,
        maxLineGap=MAX_LINE_GAP,
    )
    if raw_lines is None:
        return []

    candidates = []
    for line in raw_lines[:, 0, :]:
        x1, y1, x2, y2 = map(int, line)
        angle = line_angle_deg(x1, y1, x2, y2)
        abs_angle = abs(angle)
        if MIN_ABS_ANGLE_DEG <= abs_angle <= MAX_ABS_ANGLE_DEG:
            candidates.append((x1, y1, x2, y2, angle))
    return candidates


cv_results = {}
for name, section in sections.items():
    image = seismic_to_uint8(section)
    edges = edge_map(image)
    lines = detect_candidate_lines(edges)
    cv_results[name] = {'image': image, 'edges': edges, 'lines': lines}
    print(f'{name}: {len(lines)} линейных кандидатов')

## 4. Визуальная проверка

In [ ]:
def draw_candidates(image: np.ndarray, lines) -> np.ndarray:
    overlay = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    for x1, y1, x2, y2, _ in lines:
        # Цвет нужен только для визуального наложения в RGB.
        cv2.line(overlay, (x1, y1), (x2, y2), (255, 0, 0), 2)
    return overlay


fig, axes = plt.subplots(len(sections), 3, figsize=(18, 12), constrained_layout=True)

for row, (name, result) in enumerate(cv_results.items()):
    axes[row, 0].imshow(result['image'], cmap='gray', aspect='auto')
    axes[row, 0].set_title(f'{name}: входное изображение')

    axes[row, 1].imshow(result['edges'], cmap='gray', aspect='auto')
    axes[row, 1].set_title('Canny edges')

    axes[row, 2].imshow(draw_candidates(result['image'], result['lines']), aspect='auto')
    axes[row, 2].set_title('Отфильтрованные линии Хафа')

    for ax in axes[row]:
        ax.set_xlabel('Трасса')
        ax.set_ylabel('Временной отсчёт')

plt.show()

## Что этот результат означает

Преобразование Хафа ищет прямолинейные контуры в изображении и не знает ничего о геологии. Фильтр по углу
подавляет почти горизонтальные отражающие горизонты, но среди оставшихся линий всё равно будут ложные срабатывания.
Такой pipeline можно использовать как baseline или как генератор кандидатов для последующей ручной разметки.

### Если возвращаться к нейросети

Для осмысленного CNN/segmentation-эксперимента сначала нужно заново создать набор изображений и масок/меток,
разделить его на train/validation/test **по пространственно разнесённым участкам куба**, а затем сравнивать модель
с этим простым baseline. Без сохранённой разметки метрики старой модели воспроизвести нельзя.